# Alueellisen sähköverkon kuorman ja käyttökatkojen profilointi PROC CHART -proseduurilla

## Tiivistelmä

Verkko-operaatioiden analyytikko käyttää PROC CHART -proseduuria profiloidakseen synteettisen otoksen syöttöjohtopiirien mittarilukemia neljältä palvelualueelta ja neljästä tuotantolähteestä. Muistikirja käy läpi pystypylväs-, vaakapylväs-, lohko- ja piirakkakaaviot tiivistääkseen tuotantojakauman, vertaillakseen keskimääräistä ja kokonaiskuormaa alueittain, paljastaakseen illan kysyntähuipun tunneittain ja järjestääkseen käyttökatkominuutit lähteittäin — juuri sellaista nopeaa, tekstipohjaista tarkastelua, jota energia- ja yleishyödyllisten palvelujen tiimi ajaa ennen graafiseen raporttiin sitoutumista. DATA-askel pyytää 1 200 riviä; tämä muistikirja renderöitiin lisensoimattomassa tilassa, joka rajaa työaineiston ensimmäisiin 100 lukemaan, joten jokainen alla oleva kaavio tiivistää tuon 100 rivin otoksen.

## Tietolähteet

| Aineisto | Rivit | Kuvaus |
|---------|------|-------------|
| `grid_ops` | 100 (synteettinen otos) | Yksi rivi kutakin alueellisen sähköverkon syöttöjohtopiirin mittarilukemaa kohti, generoitu koodissa komennoilla `call streaminit(20260531)` ja `rand()`. DATA-askeleen silmukka pyytää 1 200 lukemaa, mutta lisensoimaton tila rajaa tallennetun aineiston 100 havaintoon, joten alla olevat kaaviot kuvaavat noita 100 riviä. |

# Alueellisen sähköverkon kuorman ja käyttökatkojen profilointi PROC CHART -proseduurilla

PROC CHART tuottaa merkkipohjaisia pylväs-, lohko- ja piirakkakaavioita suoraan listaukseen — ODS Graphics -laitetta ei tarvita. Tämä tekee siitä nopean ensimmäisen vaiheen tarkastelutyökalun verkko-operaatiotiimille, joka haluaa *nähdä* kuorma- ja luotettavuusaineistonsa muodon ennen viimeisteltyjen GCHART- tai SGPLOT-visualisointien rakentamista.

Tässä muistikirjassa me:

1. Generoimme synteettisen kuukauden syöttöjohtopiirien mittarilukemia alueelliselle verkko-operaattorille.
2. Kaavioimme **tuotantojakauman** (lukemien osuus lähteittäin).
3. Vertailemme **keskimääräistä ja kokonaistoimitettua kuormaa** palvelualueiden välillä.
4. Paljastamme **illan kysyntähuipun** vuorokaudenajan tunnin mukaan.
5. Käytämme **lohkokaaviota** ristiintaulukoidaksemme alueen tuotantolähdettä vasten.
6. Järjestämme **käyttökatkominuutit** lähteittäin löytääksemme epäluotettavimman syötön.

Jokainen alla oleva lause ja optio on vakiomuotoista SAS 9.4:n PROC CHART -syntaksia.

## Vaihe 1 — Generoi synteettinen operaatiodata

Alla oleva DATA-askel valmistaa mittarilukemia 1 200 iteraation silmukassa. Kullekin lukemalle määritetään palvelualue, tuotantolähde (kaasu dominoi, ja tuuli, aurinko ja vesi muodostavat loput) sekä vuorokaudenajan tunti. Kuorma on korkeampi klo 17.00–21.00 illan ikkunassa (ja merkitsemme nämä lukemat huippulukemiksi), ja arvomme käyttökatkominuutit Poisson-jakaumasta. `call streaminit` kiinnittää satunnaissiemenluvun, joten data on toistettavissa.

Lokin NOTE osoittaa käytännön tuloksen: tämä ajo on lisensoimattomassa tilassa, joten tallennettu `grid_ops`-aineisto rajataan noiden lukemien ensimmäisiin 100:aan (100 riviä, 7 saraketta). Jokainen seuraava kaavio tiivistää tuon 100 rivin otoksen, ja jokainen PROC CHART -loki vahvistaa, että se luki 100 riviä.

In [1]:
/* Synthetic feeder-circuit operations for a regional grid operator */
TIEDOT grid_ops;
    CALL streaminit(20260531);
    PITUUS region $12 source $9 peak_flag $3;
    TAULUKKO regions[4] $12 _temporary_
        ('North','South','East','West');
    TEE meter_id = 1 ASTI 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        JOS u < 0.42 NIIN source = 'Gas';
        MUUTEN JOS u < 0.70 NIIN source = 'Wind';
        MUUTEN JOS u < 0.88 NIIN source = 'Solar';
        MUUTEN source = 'Hydro';
        hour = floor(24 * rand('uniform'));
        BASE = 18 + 5 * (region = 'North') + 3 * (region = 'West');
        JOS hour >= 17 AND hour <= 21 NIIN TEE;
            load_mwh = BASE + 12 + 6 * rand('normal');
            peak_flag = 'Yes';
        LOPPU;
        MUUTEN TEE;
            load_mwh = BASE + 4 * rand('normal');
            peak_flag = 'No';
        LOPPU;
        JOS load_mwh < 0 NIIN load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        TULOSTE;
    LOPPU;
    POISTA r u BASE;
SUORITA;

NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds


## Vaihe 2 — Tuotantojakauma

Minkä osuuden lukemista kukin tuotantolähde muodostaa? Pystypylväskaavio optiolla `TYPE=PERCENT` vastaa tähän suoraan: pylväiden korkeudet ovat niiden kaikkien havaintojen prosenttiosuus, jotka osuvat kuhunkin lähdeluokkaan. Koska `source` on merkkimuuttuja, PROC CHART käsittelee sen arvot automaattisesti erillisinä luokkina.

In [2]:
PROSEDUURI chart TIEDOT=grid_ops;
    VBAR source / type=PERCENT;
SUORITA;


Percent of SOURCE

         |              *****       
         |              *****       
   40.00 +              *****       
         |              *****       
         |              *****       
   30.00 +              *****       
         |       *****  *****       
         |       *****  *****       
   20.00 +       *****  *****       
         |       *****  *****       
         |*****  *****  *****       
         |*****  *****  *****  *****
   10.00 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          Hydro  Wind    Gas   Solar
                    SOURCE


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Vaihe 3 — Keskimääräinen toimitettu kuorma alueittain

Nyt siirrymme laskennasta mittauksen tiivistämiseen. `load_mwh`:n nimeäminen optiossa `SUMVAR=` yhdessä `TYPE=MEAN`:n kanssa tekee pylvään pituudesta alueen keskimääräisen kuorman, ja pyydämme tilastosarakkeet nimenomaisesti: `MEAN` tulostaa keskiarvon kunkin pylvään viereen ja `CFREQ` lisää kertymäfrekvenssisarakkeen. Pohjoisella on korkein keskimääräinen toimitettu kuorma (keskiarvo noin 28,0 MWh), yhdenmukaisesti dataan sisäänrakennetun aluepoikkeaman kanssa, kun taas etelä on matalin (noin 19,8 MWh).

Välitämme myös `ASCENDING`-option aikoen järjestää pylväät pienimmästä suurimpaan keskiarvoon. Huomaa tulosteessa, että pylväitä **ei** ole järjestetty uudelleen — ne esiintyvät luokkajärjestyksessä (länsi, etelä, itä, pohjoinen), keskiarvoilla 24,2, 19,8, 21,7, 28,0, jotka eivät kulje lyhimmästä pisimpään. Pylväiden uudelleenjärjestämistä kaavion tilastoluvun mukaan ei ole vielä kytketty tähän PROC CHART -toteutukseen, joten `ASCENDING`/`DESCENDING` hyväksytään mutta ne eivät tällä hetkellä vaikuta mihinkään; lue järjestys tulostetusta `Mean`-sarakkeesta.

In [3]:
PROSEDUURI chart TIEDOT=grid_ops;
    HBAR region / SUMVAR=load_mwh type=mean
                  cfreq mean ascending;
SUORITA;


Mean of REGION

REGION                                                  Mean           N     Percent
                                                                                    
  West  **********************************             24.17       24.17       23.00
 South  ****************************                   19.80       43.97       21.00
  East  *******************************                21.72       65.69       29.00
 North  ****************************************       28.03       93.72       27.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Vaihe 4 — Kokonaiskuorma alueittain

Tässä `TYPE=SUM` tekee kustakin pylväästä alueen *kokonaistoimitetun* kuorman keskiarvon sijaan, joten korkeammat pylväät merkitsevät alueet, jotka toimittavat eniten kokonaisenergiaa otetuissa lukemissa. Pohjoinen johtaa jälleen kokonaistoimitetussa kuormassa.

Lause pyytää myös `SUBGROUP=peak_flag`-option, jolla PROC CHART jakaa kunkin pylvään sen mukaan, osuiko lukema illan huippuikkunaan. SAS:ssa tämä segmentoi kunkin pylvään erillisellä symbolilla alaryhmätasoa kohti ja tulostaa selitteen. Tämä toteutus ei vielä renderöi alaryhmäsegmentointia (dokumentoitu ominaisuuspuute), joten pylväät tulostuvat kiinteinä `*`-jaksoina ilman segmenttikohtaista erittelyä — pylväiden *summat* ovat oikein, mutta huippu-vs-huipun ulkopuolella -jakoa ei näytetä tässä. Nähdäksesi kuinka paljon kuormasta osuu huipputunneille, käytä vuorokaudenajan näkymää vaiheessa 5.

In [4]:
PROSEDUURI chart TIEDOT=grid_ops;
    VBAR region / SUMVAR=load_mwh type=sum
                  SUBGROUP=peak_flag;
SUORITA;


Sum of REGION

         |                     *****
         |                     *****
         |                     *****
     600 +              *****  *****
         |*****         *****  *****
         |*****         *****  *****
         |*****         *****  *****
     400 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
     200 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          West   South  East   North
                    REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Vaihe 5 — Kuorman muoto päivän aikana

`hour` on jatkuva, joten kiinnitämme luokittelun itse eksplisiittisellä `MIDPOINTS=`-listalla 4 tunnin keskipisteissä (2, 6, 10, 14, 18, 22). Kukin pylväs näyttää keskimääräisen toimitetun kuorman tuota tuntia lähellä oleville lukemille. Pylvään, jonka keskipiste on 18, tulisi erottua — se on illan kysyntähuippu, jonka rakensimme dataan.

In [5]:
PROSEDUURI chart TIEDOT=grid_ops;
    VBAR hour / SUMVAR=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
SUORITA;


Mean of HOUR

         |                            *****       
         |                            *****       
   30.00 +                            *****       
         |                            *****  *****
         |                            *****  *****
         |              *****         *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                            HOUR


NOTE: PROC CHART data=grid_ops


## Vaihe 6 — Alue tuotantolähteen mukaan (lohkokaavio)

BLOCK-kaavio renderöi pienen kaksisuuntaisen taulukon lohkojen ruudukkona. Optioilla `GROUP=source` ja `SUMVAR=load_mwh / TYPE=MEAN` kaavio ristiintaulukoi kunkin alueen sitä palvelevaa tuotantolähdettä vasten, lohkon korkeuden ollessa suhteessa keskimääräiseen kuormaan — tiivis tapa havaita, mitkä alue/lähde-yhdistelmät kantavat raskaimman keskimääräisen kuorman.

In [6]:
PROSEDUURI chart TIEDOT=grid_ops;
    BLOCK region / SUMVAR=load_mwh type=mean
                   GROUP=source;
SUORITA;


Block chart of Mean of REGION by SOURCE

                          /####\
  /####\                  /####\
  /####\          /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
   West   South    East   North 
             REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Vaihe 7 — Tuotantojakauma piirakkakaaviona

Sama lähdeosuustieto vaiheesta 2, piirretty piirakkana. PIE optiolla `TYPE=PERCENT` mitoittaa kunkin sektorin sen prosenttiosuudella kokonaislukemista ja tulostaa sektoriprosenttien selitteen kuvan viereen.

In [7]:
PROSEDUURI chart TIEDOT=grid_ops;
    PIE source / type=PERCENT;
SUORITA;


Pie chart of SOURCE

          SOURCE     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
           Hydro       14.00     14.0%  ####
            Wind       28.00     28.0%  ********
             Gas       45.00     45.0%  ++++++++++++++
           Solar       13.00     13.0%  OOOO


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Vaihe 8 — Käyttökatkominuutit lähteittäin

Lopuksi järjestämme luotettavuuden. `SUMVAR=outage_min` yhdessä `TYPE=SUM`:n kanssa summaa käyttökatkominuutit lähdettä kohti. Välitämme `DESCENDING`-option yrittääksemme nostaa huonoiten suoriutuvan lähteen ylös, mutta kuten vaiheessa 3 pylväitä ei järjestetä uudelleen — ne tulostuvat luokkajärjestyksessä (vesi, tuuli, kaasu, aurinko) eikä pylväiden uudelleenjärjestämistä ole vielä toteutettu. Lue järjestys tulostetusta `Sum`-sarakkeesta: kaasu muodostaa eniten kokonaiskäyttökatkominuutteja tässä otoksessa (122), selvästi tuulen (64), auringon (43) ja veden (38) edellä. Tämä seuraa tuotantojakaumaa — kaasu palvelee eniten lukemia, joten se kerää eniten käyttökatkominuutteja kaikkiaan.

In [8]:
PROSEDUURI chart TIEDOT=grid_ops;
    HBAR source / SUMVAR=outage_min type=sum LASKEVA;
SUORITA;


Sum of SOURCE

SOURCE                                                   Sum        Cum.     Percent
                                                                     Sum            
 Hydro  ************                                      38          38       14.00
  Wind  *********************                             64         102       28.00
   Gas  ****************************************         122         224       45.00
 Solar  **************                                    43         267       13.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Tulosten tulkinta

Kaavioiden lukeminen yhdessä antaa operaatiotiimille nopean tilannekuvan (tämän ajon tallentaman 100 rivin otoksen yli):

- **Tuotantojakauma (vaiheet 2 ja 7).** Kaasu kantaa suurimman osuuden lukemista (noin 45 %), tuuli toisena (noin 28 %) sekä vesi ja aurinko perässä (noin 14 % ja 13 %) — pystypylväs ja piirakka kertovat saman tarinan kahdella tavalla, hyödyllinen tarkistus.
- **Kuorma alueittain (vaiheet 3 ja 4).** Keskikuorman HBAR näyttää pohjoisen ajavan korkeinta keskimääräistä toimitettua kuormaa (keskiarvo ~28 MWh) ja etelän matalinta (~20 MWh), yhdenmukaisesti dataan sisäänrakennetun aluepoikkeaman kanssa. SUM-kaavio vahvistaa, että pohjoinen toimittaa myös eniten kokonaisenergiaa.
- **Päivittäinen kuorman muoto (vaihe 5).** Tuntiin 18 keskitetty keskipistepylväs on selvästi korkein, mikä vahvistaa dataan rakennetun klo 17.00–21.00 kysyntähuipun — juuri sinne, mihin yleishyödyllinen palvelu keskittäisi kysyntäjouston ja kapasiteettisuunnittelun.
- **Luotettavuus (vaihe 8).** Käyttökatkominuuttien summaaminen lähteittäin nostaa esiin kaasun tämän otoksen suurimpana käyttökatkojen aiheuttajana (122 minuuttia), luontevana seuraavana kohteena kunnossapidon tarkastelulle — vaikka tämä heijastaa lähinnä sitä, että kaasu palvelee eniten lukemia.

Kaksi tässä käytettyä näyttöoptiota — `ASCENDING`/`DESCENDING`-pylväiden uudelleenjärjestäminen (vaiheet 3 ja 8) ja `SUBGROUP=`-pylväiden segmentointi (vaihe 4) — hyväksytään jäsentimessä mutta niitä ei vielä renderöidä tässä PROC CHART -toteutuksessa, joten järjestykset ja jaot luetaan tulostetuista tilastosarakkeista pylväsjärjestyksen tai varjostuksen sijaan.

PROC CHART on vain merkkitulostetta, joten johtoryhmävalmiita visuaaleja varten tiimi siirtäisi nämä samat näkymät PROC GCHART- tai PROC SGPLOT -proseduuriin. Mutta nollavalmisteluvaiheen ensitarkasteluna tuoreesta poiminnasta nämä kaaviot vastaavat operatiivisiin kysymyksiin — jakauma, suuruus ja ajoitus — sekunneissa.